# Figures for the DL_SS26 project

**Audience:** anyone who wants to inspect or re-generate the figures used in the presentation and the summary document.

**Goal:** produce six slide-quality PNGs that visualize the dataset, the holdout-test performance of all five models, the cross-input experiment, and the hyperparameter-search behavior.

**How to use:** run the cells top-to-bottom. The Setup cell loads metrics + predictions; each figure cell is independent after that and can be re-run in isolation to iterate on style. Outputs go to `results/figures/`.

**Canonical equivalent:** `scripts/make_figures.py` produces the same six PNGs without Jupyter. This notebook is the documented walk-through; the script is the reproducibility entry point.

## Figure inventory

| # | File | Purpose |
|---|------|---------|
| 1 | `fig1_roc_curves.png` | ROC curves for all 6 models on the 20% holdout |
| 2 | `fig2_auc_with_ci.png` | Holdout AUC with bootstrapped 95% CIs |
| 3 | `fig3_confusion_matrices.png` | Confusion matrices at threshold 0.5 |
| 4 | `fig4_cross_input_grid.png` | Architecture × input heatmap (the *input vs architecture* experiment) |
| 5 | `fig5_optuna_progress.png` | TPE vs Random running-max AUC during search |
| 6 | `fig6_holdout_class_balance.png` | Dataset composition: MSS vs MSI-H counts per split |

## Data dependencies (all in `results/`)

- `holdout_<model>.json` — point estimates + 95% CIs (6 files, one per model)
- `holdout_<model>_predictions.npz` — per-sample `y_true` + `y_prob` (6 files)
- `cross_input_summary.json` — 3×4 grid of architecture × input combinations
- `optuna_studies.db` — SQLite Optuna database, contains hyperparameter trials

## Setup

Loads each model's holdout metrics and per-sample predictions. Prints a one-line summary so you can sanity-check the data before plotting. Defines the canonical model ordering and color palette used across all figures (consistent colors help the eye link models across panels).

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# Repo paths — notebook lives in scripts/, so repo root is one level up
REPO = Path('..').resolve()
RESULTS = REPO / 'results'
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

# Canonical display order: worst -> best holdout AUC.
# Color is consistent per model across all figures.
MODELS = [
    ('logreg_epic',        'Logreg / EPIC (8)',          'tab:gray'),
    ('lstm_chromorder',    'LSTM / chr-ordered',         'tab:purple'),
    ('cnn_chromorder',     'CNN / chr-ordered',          'tab:orange'),
    ('mlp_pathways',       'MLP / pathways (50)',        'tab:green'),
    ('transformer_panel',  'Transformer / panel (38)',   'tab:red'),
    ('mlp_chrord',         'MLP / chr-ordered (tuned)',  'tab:blue'),
]

data = {}
for slug, label, color in MODELS:
    with (RESULTS / f'holdout_{slug}.json').open() as f:
        metrics = json.load(f)
    preds = np.load(RESULTS / f'holdout_{slug}_predictions.npz')
    data[slug] = {
        'label': label, 'color': color, 'metrics': metrics,
        'y_true': preds['y_true'], 'y_prob': preds['y_prob'],
    }
    auc = metrics['auc']
    print(f"  {label:32s} AUC {auc['point']:.3f}  [{auc['low']:.3f} - {auc['high']:.3f}]")

# Slide-friendly style: large fonts, restrained palette, no chartjunk
plt.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 14,
    'axes.labelsize': 13,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.15,
})

## Figure 1 — ROC curves

**What it shows.** True-positive rate (sensitivity, y-axis) vs false-positive rate (1 – specificity, x-axis) as the classification threshold is varied, for each of the 6 models. The dashed diagonal is what a chance classifier produces.

**How to read it.** Curves toward the top-left = stronger discrimination. A curve that hugs the upper-left corner reflects high sensitivity at low FPR.

**Why include it.** ROC curves are the de-facto visual standard for binary classifiers and let viewers compare 6 models on identical axes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))

for slug, info in data.items():
    fpr, tpr, _ = roc_curve(info['y_true'], info['y_prob'])
    auc = info['metrics']['auc']
    label = (f"{info['label']:30s} "
             f"AUC = {auc['point']:.3f} "
             f"[{auc['low']:.2f}\u2013{auc['high']:.2f}]")
    ax.plot(fpr, tpr, color=info['color'], lw=2.3, label=label)

ax.plot([0, 1], [0, 1], color='black', linestyle='--', lw=1, alpha=0.4,
        label='chance (AUC = 0.50)')
ax.set_xlabel('False positive rate (1 \u2013 specificity)')
ax.set_ylabel('True positive rate (sensitivity)')
ax.set_title('ROC curves on 20% holdout test set (n = 100)')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect('equal')
ax.grid(alpha=0.3, linestyle=':')
leg = ax.legend(loc='lower right', frameon=True,
                prop={'family': 'monospace', 'size': 9.5})
leg.get_frame().set_alpha(0.9)

fig.savefig(FIGURES / 'fig1_roc_curves.png')
plt.show()

## Figure 2 — Holdout AUC with 95% bootstrap CI

**What it shows.** The point-estimate holdout AUC for each model, sorted ascending, with error bars showing the 95% bootstrap confidence interval (2000 resamples of the 100-sample test set).

**How to read it.** Longer bars = better discrimination. Overlapping CI ranges between two models mean their difference is not statistically distinguishable at this sample size — i.e. the empirical ranking could shift with a different 100-sample test set.

**Why include it.** This is the single most informative figure of the project: in one image a viewer sees both the overall ranking and the statistical uncertainty around it. Headline finding (MLP-on-chr-ordered beats transformer) is visible at a glance.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ordered = sorted(data.items(), key=lambda kv: kv[1]['metrics']['auc']['point'])
labels = [info['label'] for _, info in ordered]
aucs   = [info['metrics']['auc']['point'] for _, info in ordered]
lows   = [info['metrics']['auc']['low']   for _, info in ordered]
highs  = [info['metrics']['auc']['high']  for _, info in ordered]
colors = [info['color'] for _, info in ordered]

lower_err = [a - lo for a, lo in zip(aucs, lows)]
upper_err = [hi - a for a, hi in zip(aucs, highs)]
y_pos = np.arange(len(labels))

ax.barh(y_pos, aucs, xerr=[lower_err, upper_err], color=colors,
        edgecolor='black', linewidth=0.5, capsize=4, alpha=0.85)
ax.axvline(0.5, color='gray', linestyle='--', lw=1.2, alpha=0.6)
ax.text(0.5, len(labels) - 0.4, ' chance', color='gray', fontsize=10, va='center')

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('AUROC on holdout test (n = 100) \u2014 95% bootstrap CI')
ax.set_xlim(0.3, 1.05)
ax.set_title('Holdout-AUC by model')
ax.grid(axis='x', alpha=0.3, linestyle=':')

for i, (a, hi) in enumerate(zip(aucs, highs)):
    ax.text(hi + 0.012, i, f'{a:.3f}', va='center', fontsize=11)

fig.savefig(FIGURES / 'fig2_auc_with_ci.png')
plt.show()

## Figure 3 — Confusion matrices

**What it shows.** For each model, the 2×2 table of (actual MSS or MSI-H) × (predicted MSS or MSI-H) using the conventional classification threshold of 0.5.

**How to read it.** The diagonal (top-left + bottom-right) is correct predictions; off-diagonal is errors. The bottom-right cell — the count of MSI-H tumors correctly identified — is the clinically relevant one, since MSI-H tumors are candidates for immunotherapy.

**Why include it.** Complements AUC by showing what each model actually does at a threshold a clinician would use. Reveals e.g. that the LSTM has high false-negative rate (16 of 32 MSI-H missed), which the AUC alone hides.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

ordered = sorted(data.items(), key=lambda kv: kv[1]['metrics']['auc']['point'])

for ax, (slug, info) in zip(axes, ordered):
    cm = np.array(info['metrics']['confusion_matrix'])
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=cm.max())
    threshold = cm.max() * 0.6
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > threshold else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=18, color=color, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['MSS', 'MSI-H'])
    ax.set_yticklabels(['MSS', 'MSI-H'])
    ax.set_xlabel('predicted')
    ax.set_ylabel('actual')
    auc = info['metrics']['auc']['point']
    ax.set_title(f"{info['label']}\nAUC = {auc:.3f}", fontsize=11)

fig.suptitle('Confusion matrices on holdout test set (threshold = 0.5)',
             fontsize=14, y=1.00)
fig.tight_layout()
fig.savefig(FIGURES / 'fig3_confusion_matrices.png')
plt.show()

## Figure 4 — Cross-input grid: architecture × input

**What it shows.** Heatmap of holdout AUC where rows are 3 architectures (logreg / MLP / transformer) and columns are 4 input formats (EPIC fractions, pathway scores, the 38-gene panel, chr-ordered 12k genes). The diagonal-ish pattern shows each architecture's performance on its 'natural' input AND on inputs natural to other architectures.

**How to read it.** Cells are colored by AUC (red = chance, green = near-perfect). The numerical value is overlaid for precision. The dashed outline at the bottom-right marks the cell that was not evaluated (transformer × 12k genes — attention's O(n²) memory cost exceeded GPU memory at our tuned hyperparameters; adapting the architecture to fit would have introduced a confound).

**Why include it.** This is the methodologically most interesting figure. It supports the report's main finding: **the input format matters more than the architecture choice**. Reading right-to-left along any row, AUC monotonically increases as you move from sparse (8 EPIC features) to dense (12k chr-ordered genes) inputs.

In [ ]:
with (RESULTS / 'cross_input_summary.json').open() as f:
    grid_data = json.load(f)

archs        = ['logreg', 'mlp', 'transformer']
inputs       = ['epic', 'pathways', 'panel38', 'chrord12k']
arch_labels  = ['Logreg', 'MLP', 'Transformer']
input_labels = ['EPIC (8)', 'Pathways (50)', 'Panel (38)', 'ChrOrd (12k)']

grid_auc = np.full((len(archs), len(inputs)), np.nan)
for ai, arch in enumerate(archs):
    for ii, inp in enumerate(inputs):
        cell = f'{arch}_on_{inp}'
        if cell in grid_data:
            grid_auc[ai, ii] = grid_data[cell]['auc']['point']

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(grid_auc, cmap='RdYlGn', vmin=0.45, vmax=1.0, aspect='auto')

for ai in range(len(archs)):
    for ii in range(len(inputs)):
        v = grid_auc[ai, ii]
        if np.isnan(v):
            ax.text(ii, ai, 'n/a', ha='center', va='center', fontsize=12,
                    color='gray', style='italic')
        else:
            text_color = 'white' if (v < 0.62 or v > 0.93) else 'black'
            ax.text(ii, ai, f'{v:.3f}', ha='center', va='center',
                    fontsize=15, color=text_color, fontweight='bold')

ax.set_xticks(range(len(inputs)))
ax.set_xticklabels(input_labels)
ax.set_yticks(range(len(archs)))
ax.set_yticklabels(arch_labels)
ax.set_xlabel('Input format')
ax.set_ylabel('Architecture')
ax.set_title('Cross-input grid: holdout AUC by architecture \u00d7 input')

cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
cbar.set_label('AUROC', rotation=270, labelpad=18)

if np.isnan(grid_auc[2, 3]):
    ax.add_patch(plt.Rectangle((2.5, 1.5), 1, 1, fill=False,
                                edgecolor='black', lw=1.5,
                                linestyle='--', alpha=0.6))

fig.savefig(FIGURES / 'fig4_cross_input_grid.png')
plt.show()

## Figure 5 — Optuna search progress

**What it shows.** For each of the four cluster-based hyperparameter searches, the running maximum AUC observed across completed trials. Two curves per panel: TPE (Bayesian, blue) and Random (gray).

**How to read it.** A faster-rising curve means the sampler found good configurations earlier. If TPE rises faster than Random, that's evidence the informed search was worthwhile; if both saturate quickly at similar values, the search reached the metric's resolution floor and the difference between samplers is meaningless.

**Why include it.** Methodological evidence that the tuning was both rigorous (we compared informed vs random) and complete (search saturated, more trials wouldn't help).

**Note.** The MLP/pathways search was run locally and its Optuna database was not merged into the cluster's SQLite store. It's omitted from this panel; its result still appears in figures 1–4. The other four models cover the bulk of the search effort (1000 + 300 + 300 + 200 trials respectively for the TPE branches).

In [ ]:
import optuna

storage = f"sqlite:///{RESULTS / 'optuna_studies.db'}"

studies = [
    ('CNN / chr-ordered',      'cnn_tpe_v2',         'cnn_random_v2'),
    ('Transformer / panel-38', 'transformer_tpe_v2', 'transformer_random_v2'),
    ('LSTM / chr-ordered',     'lstm_tpe_v2',        'lstm_random_v2'),
    ('MLP / chr-ordered',      'mlp_chrord_tpe_v2',  'mlp_chrord_random_v2'),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
axes = axes.flatten()

for ax, (title, tpe_name, rnd_name) in zip(axes, studies):
    for study_name, color, label in [(tpe_name, 'tab:blue', 'TPE'),
                                     (rnd_name, 'tab:gray', 'Random')]:
        try:
            st = optuna.load_study(study_name=study_name, storage=storage)
            completed = sorted(
                (t for t in st.trials if t.state == optuna.trial.TrialState.COMPLETE),
                key=lambda t: t.number,
            )
            if not completed:
                continue
            trial_nums = [t.number for t in completed]
            values = [t.value for t in completed]
            running_max = np.maximum.accumulate(values)
            ax.plot(trial_nums, running_max, color=color, lw=2.2,
                    label=label, drawstyle='steps-post')
        except Exception as exc:
            print(f'  warn: {study_name}: {exc}')

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('completed trial')
    ax.set_ylabel('best AUC so far')
    ax.grid(alpha=0.3, linestyle=':')
    ax.legend(loc='lower right', fontsize=10)

fig.suptitle('Optuna search progress: TPE vs Random '
             '(running max AUC across completed trials)',
             fontsize=14, y=1.00)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(FIGURES / 'fig5_optuna_progress.png')
plt.show()

## Figure 6 — Holdout split composition

**What it shows.** Counts of MSS vs MSI-H samples in the 80% holdout-train and 20% holdout-test sets. Demonstrates that the stratified split preserves the ~2:1 MSS:MSI-H ratio in both portions.

**Why include it.** A simple dataset-composition figure. Reassures viewers that the holdout split is balanced and that no test fold accidentally received an extreme class distribution that would skew the holdout numbers.

In [ ]:
y_test = data['mlp_chrord']['y_true']
n_pos_test = int(y_test.sum())
n_neg_test = len(y_test) - n_pos_test

n_pos_train = 127  # 159 total MSI-H \u00d7 0.80 (matches scripts/01_make_splits)
n_neg_train = 272

fig, ax = plt.subplots(figsize=(7.5, 4))
splits = ['Holdout-train (80%)', 'Holdout-test (20%)']
mss_counts   = [n_neg_train, n_neg_test]
msih_counts  = [n_pos_train, n_pos_test]

x = np.arange(len(splits))
width = 0.4

b1 = ax.bar(x - width / 2, mss_counts, width,
            label='MSS',   color='tab:blue',   alpha=0.85, edgecolor='black', linewidth=0.5)
b2 = ax.bar(x + width / 2, msih_counts, width,
            label='MSI-H', color='tab:orange', alpha=0.85, edgecolor='black', linewidth=0.5)
for bars, vals in [(b1, mss_counts), (b2, msih_counts)]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 5, str(v),
                ha='center', va='bottom', fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel('number of samples')
ax.set_title('Holdout split composition  (n = 499 total samples)')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3, linestyle=':')
ax.set_ylim(0, max(mss_counts) * 1.18)

fig.savefig(FIGURES / 'fig6_holdout_class_balance.png')
plt.show()